# **뉴시티 + 검단 통합 데이터 — 선형 회귀 모델 (계약연도 기준 분할)**

### **0. 라이브러리 임포트**

In [218]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정
matplotlib.rc('font', family='Malgun Gothic')
matplotlib.rc('axes', unicode_minus=False)

### **1. 데이터 로드 및 통합**


In [219]:
# ── 데이터 불러오기 ──
df = pd.read_csv(r"C:\3_1_DataMining\팀 프로젝트\신도시 데이터\real_new_city.csv",
                          encoding='utf-8-sig')


print(f"통합 데이터 건수: {len(df)}개")
print(f"도시 목록: {df['도시명'].unique()}")

통합 데이터 건수: 126375개
도시 목록: <StringArray>
['청라', '송도', '판교', '광교', '운정', '검단']
Length: 6, dtype: str


### **2. 데이터 전처리**

#### **2-1. 거래금액 숫자 변환**

In [220]:
# ── 거래금액 전처리 ──
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)

#### **2-2. ㎡당 가격 파생변수 생성 (타겟변수)**

In [221]:
# ── 타겟 변수 생성 ──
df['m2당가격'] = df['거래금액(만원)'] / df['전용면적(㎡)']

#### **2-3. 발표후경과년수 3 미만 제거**

In [222]:
# ── 발표후경과년수 필터링 ──
before = len(df)
df = df[df['발표후경과년수'] >= 3]
print(f"발표후경과년수 3 미만 제거: {before - len(df)}개 → 남은 데이터: {len(df)}개")

발표후경과년수 3 미만 제거: 3863개 → 남은 데이터: 122512개


#### **2-4. 전용면적 33㎡ 미만 제거**

In [223]:
# ── 전용면적 필터링 ──
before = len(df)
df = df[df['전용면적(㎡)'] >= 33]
print(f"전용면적 33㎡ 미만 제거: {before - len(df)}개 → 남은 데이터: {len(df)}개")

전용면적 33㎡ 미만 제거: 291개 → 남은 데이터: 122221개


#### **2-5. 이상치 제거 (z-score 기준)**

In [224]:
# ── z-score 이상치 제거 ──
before = len(df)

mean = df['m2당가격'].mean()
std  = df['m2당가격'].std()
z_scores = (df['m2당가격'] - mean) / std
df = df[z_scores.abs() <= 2]

print(f"z-score 이상치 제거: {before - len(df)}개 → 남은 데이터: {len(df)}개")

z-score 이상치 제거: 5286개 → 남은 데이터: 116935개


### **3. 입력 변수 및 타겟 변수 설정**


In [225]:
# ── 변수 목록 설정 ──
features = ['건축년도', '층',
            '지하철호선개수', '기차역까지의거리',
            '가장 가까운 지하철역까지의 거리', '가장 가까운 IC와의 거리',
            '발표후경과년수', 'CPI', '계약연도', '서울도심거리',
            '단지별_세대수', '도시별_세대수']

# ── 결측치 제거 ──
before = len(df)
df = df.dropna(subset=features + ['m2당가격'])
print(f"결측치 제거: {before - len(df)}개 → 남은 데이터: {len(df)}개")

결측치 제거: 6개 → 남은 데이터: 116929개


### **4. 훈련/테스트 세트 분할 (계약연도 기준)**


In [226]:
# ── 계약연도 기준 분할 ──
df_train = df[df['계약연도'] <= 2023].copy()
df_test  = df[df['계약연도'] >= 2024].copy()

train_input  = df_train[features]
train_target = df_train['m2당가격']
test_input   = df_test[features]
test_target  = df_test['m2당가격']

print(f"훈련 세트: {train_input.shape}  ")
print(f"테스트 세트: {test_input.shape}  ")

훈련 세트: (96811, 12)  
테스트 세트: (20118, 12)  


### **5. 표준화 (StandardScaler)**


In [227]:
# ── 표준화 ──
ss = StandardScaler()
ss.fit(train_input)

train_scaled = ss.transform(train_input)
test_scaled  = ss.transform(test_input)

print(f"train_scaled shape: {train_scaled.shape}")
print(f"test_scaled  shape: {test_scaled.shape}")



train_scaled shape: (96811, 12)
test_scaled  shape: (20118, 12)


### **6. 선형 회귀 모델 학습**

In [228]:
# ── 선형 회귀 모델 학습 ──
model = LinearRegression()
model.fit(train_scaled, train_target)




,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### **7. 테스트 결과 — 결정계수(R²) 및 오차 지표**



In [229]:
y_pred = model.predict(test_scaled)

mae  = mean_absolute_error(test_target, y_pred)
mse  = mean_squared_error(test_target, y_pred)
rmse = np.sqrt(mse)

print(f"훈련 세트 R²: {model.score(train_scaled, train_target):.4f}")
print(f"테스트 세트 R²: {model.score(test_scaled,  test_target):.4f}")
print(f"MAE : {mae:.2f} 만원/㎡")
print(f"RMSE: {rmse:.2f} 만원/㎡")

훈련 세트 R²: 0.6742
테스트 세트 R²: 0.5584
MAE : 122.42 만원/㎡
RMSE: 149.95 만원/㎡
